# 01 — Data Collection

This notebook drives the **raw-data acquisition** stage of the project:

1. Download IMDB official datasets (`.tsv.gz`).
2. Download & extract MovieLens 25M.
3. Inspect schemas, row counts, and file sizes.
4. Filter `title.basics` to feature films only.
5. Run a small TMDB enrichment sample.
6. Produce a **data-quality report** (nulls, duplicates, anomalies).

> All downloads are idempotent — re-running the notebook does not re-fetch
> files that already exist on disk.

## 0. Setup

In [1]:
# Make project root importable from a notebook in ./notebooks/
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

from config import settings, setup_logging
setup_logging()

import logging
logger = logging.getLogger("01_data_collection")

logger.info("Project root: %s", settings.PATHS.ROOT)
logger.info("Raw-data dir: %s", settings.PATHS.RAW_DATA)
logger.info("TMDB configured: %s", settings.TMDB.is_configured())

2026-05-13 21:23:29 | INFO     | 01_data_collection:12 | Project root: C:\Users\Kadir\Desktop\imdb_project
2026-05-13 21:23:29 | INFO     | 01_data_collection:13 | Raw-data dir: C:\Users\Kadir\Desktop\imdb_project\data\raw
2026-05-13 21:23:29 | INFO     | 01_data_collection:14 | TMDB configured: False


## 1. Download IMDB datasets

We pull all five official IMDB TSVs. They stay gzipped on disk — pandas reads
`.tsv.gz` directly.

In [2]:
from src.data.downloader import IMDBDownloader

imdb_files = IMDBDownloader().download_all()
for name, path in imdb_files.items():
    size_mb = path.stat().st_size / 1e6
    logger.info("%-30s %7.1f MB  ->  %s", name, size_mb, path)

2026-05-13 21:23:29 | INFO     | src.data.downloader:106 | Downloading https://datasets.imdbws.com/title.basics.tsv.gz -> C:\Users\Kadir\Desktop\imdb_project\data\raw\title.basics.tsv.gz


title.basics.tsv.gz:   0%|          | 0.00/211M [00:00<?, ?B/s]

2026-05-13 21:24:03 | INFO     | src.data.downloader:130 | Saved title.basics.tsv.gz (221.6 MB)
2026-05-13 21:24:03 | INFO     | src.data.downloader:106 | Downloading https://datasets.imdbws.com/title.ratings.tsv.gz -> C:\Users\Kadir\Desktop\imdb_project\data\raw\title.ratings.tsv.gz


title.ratings.tsv.gz:   0%|          | 0.00/8.04M [00:00<?, ?B/s]

2026-05-13 21:24:04 | INFO     | src.data.downloader:130 | Saved title.ratings.tsv.gz (8.4 MB)
2026-05-13 21:24:04 | INFO     | src.data.downloader:106 | Downloading https://datasets.imdbws.com/title.crew.tsv.gz -> C:\Users\Kadir\Desktop\imdb_project\data\raw\title.crew.tsv.gz


...

2026-05-13 21:24:17 | INFO     | src.data.downloader:130 | Saved title.crew.tsv.gz (81.5 MB)
2026-05-13 21:24:18 | INFO     | src.data.downloader:106 | Downloading https://datasets.imdbws.com/title.principals.tsv.gz -> C:\Users\Kadir\Desktop\imdb_project\data\raw\title.principals.tsv.gz


title.principals.tsv.gz:   0%|          | 0.00/730M [00:00<?, ?B/s]

2026-05-13 21:26:22 | INFO     | src.data.downloader:130 | Saved title.principals.tsv.gz (765.6 MB)
2026-05-13 21:26:22 | INFO     | src.data.downloader:106 | Downloading https://datasets.imdbws.com/name.basics.tsv.gz -> C:\Users\Kadir\Desktop\imdb_project\data\raw\name.basics.tsv.gz


name.basics.tsv.gz:   0%|          | 0.00/289M [00:00<?, ?B/s]

2026-05-13 21:27:12 | INFO     | src.data.downloader:130 | Saved name.basics.tsv.gz (303.3 MB)
2026-05-13 21:27:12 | INFO     | src.data.downloader:231 | IMDB download complete: 5 files in C:\Users\Kadir\Desktop\imdb_project\data\raw
2026-05-13 21:27:12 | INFO     | 01_data_collection:6 | title.basics.tsv.gz              221.6 MB  ->  C:\Users\Kadir\Desktop\imdb_project\data\raw\title.basics.tsv.gz
2026-05-13 21:27:12 | INFO     | 01_data_collection:6 | title.ratings.tsv.gz               8.4 MB  ->  C:\Users\Kadir\Desktop\imdb_project\data\raw\title.ratings.tsv.gz
2026-05-13 21:27:12 | INFO     | 01_data_collection:6 | title.crew.tsv.gz                 81.5 MB  ->  C:\Users\Kadir\Desktop\imdb_project\data\raw\title.crew.tsv.gz
2026-05-13 21:27:12 | INFO     | 01_data_collection:6 | title.principals.tsv.gz          765.6 MB  ->  C:\Users\Kadir\Desktop\imdb_project\data\raw\title.principals.tsv.gz
2026-05-13 21:27:12 | INFO     | 01_data_collection:6 | name.basics.tsv.gz               30

## 2. Download MovieLens 25M

This is a ~250 MB zip; first run takes a few minutes. Subsequent runs are
free (skip-if-extracted).

In [3]:
from src.data.downloader import MovieLensDownloader

ml_dir = MovieLensDownloader().download_and_extract()
logger.info("MovieLens extracted at: %s", ml_dir)
for csv in sorted(ml_dir.glob("*.csv")):
    logger.info("  %-15s %7.1f MB", csv.name, csv.stat().st_size / 1e6)

2026-05-13 21:27:12 | INFO     | src.data.downloader:106 | Downloading https://files.grouplens.org/datasets/movielens/ml-25m.zip -> C:\Users\Kadir\Desktop\imdb_project\data\raw\ml-25m.zip


ml-25m.zip:   0%|          | 0.00/250M [00:00<?, ?B/s]

2026-05-13 21:27:58 | INFO     | src.data.downloader:130 | Saved ml-25m.zip (262.0 MB)
2026-05-13 21:27:58 | INFO     | src.data.downloader:295 | Extracting ml-25m.zip -> C:\Users\Kadir\Desktop\imdb_project\data\raw
2026-05-13 21:28:09 | INFO     | src.data.downloader:310 | MovieLens ready at C:\Users\Kadir\Desktop\imdb_project\data\raw\ml-25m
2026-05-13 21:28:09 | INFO     | 01_data_collection:4 | MovieLens extracted at: C:\Users\Kadir\Desktop\imdb_project\data\raw\ml-25m
2026-05-13 21:28:09 | INFO     | 01_data_collection:6 |   genome-scores.csv   435.2 MB
2026-05-13 21:28:09 | INFO     | 01_data_collection:6 |   genome-tags.csv     0.0 MB
2026-05-13 21:28:09 | INFO     | 01_data_collection:6 |   links.csv           1.4 MB
2026-05-13 21:28:09 | INFO     | 01_data_collection:6 |   movies.csv          3.0 MB
2026-05-13 21:28:09 | INFO     | 01_data_collection:6 |   ratings.csv       678.3 MB
2026-05-13 21:28:09 | INFO     | 01_data_collection:6 |   tags.csv           38.8 MB


## 3. Schema inspection — IMDB

For each TSV we read just the **first 1000 rows** to discover the schema
cheaply. We also report the *full* row count using a streamed line count.

In [4]:
import pandas as pd

def quick_schema(path, n=5):
    df = pd.read_csv(path, sep="\t", nrows=1000, na_values="\\N",
                     low_memory=False)
    return {
        "columns": list(df.columns),
        "dtypes": {c: str(t) for c, t in df.dtypes.items()},
        "sample": df.head(n).to_dict(orient="records"),
    }

schemas = {name: quick_schema(path) for name, path in imdb_files.items()}
for name, info in schemas.items():
    print(f"\n== {name} ==")
    print(" columns:", info["columns"])


== title.basics.tsv.gz ==
 columns: ['tconst', 'titleType', 'primaryTitle', 'originalTitle', 'isAdult', 'startYear', 'endYear', 'runtimeMinutes', 'genres']

== title.ratings.tsv.gz ==
 columns: ['tconst', 'averageRating', 'numVotes']

== title.crew.tsv.gz ==
 columns: ['tconst', 'directors', 'writers']

== title.principals.tsv.gz ==
 columns: ['tconst', 'ordering', 'nconst', 'category', 'job', 'characters']

== name.basics.tsv.gz ==
 columns: ['nconst', 'primaryName', 'birthYear', 'deathYear', 'primaryProfession', 'knownForTitles']


In [5]:
# Row counts (streamed — no full read into memory)
import gzip

def gz_line_count(path):
    n = 0
    with gzip.open(path, "rb") as fh:
        for _ in fh:
            n += 1
    return n - 1   # minus header

row_counts = {name: gz_line_count(path) for name, path in imdb_files.items()}
pd.Series(row_counts, name="rows").sort_values(ascending=False).to_frame()

,rows
title.principals.tsv.gz,99422049
name.basics.tsv.gz,15322955
title.basics.tsv.gz,12497885
title.crew.tsv.gz,12497885
title.ratings.tsv.gz,1669211


## 4. Filter to feature films

`title.basics` contains every kind of title (TV, episodes, shorts…).
For our recommender we only want **movies** that survive our quality filters
(see `config.IMDBConfig`).

We use `usecols` + a `dtype` map so the full file fits in ~250 MB of RAM
rather than 1.5 GB.

In [6]:
basics_path = imdb_files["title.basics.tsv.gz"]

USECOLS = ["tconst", "titleType", "primaryTitle", "originalTitle",
           "isAdult", "startYear", "runtimeMinutes", "genres"]
DTYPES = {
    "tconst": "string",
    "titleType": "category",
    "primaryTitle": "string",
    "originalTitle": "string",
    "isAdult": "string",          # mixed values — coerce after
    "startYear": "string",
    "runtimeMinutes": "string",
    "genres": "string",
}

basics = pd.read_csv(
    basics_path, sep="\t", na_values="\\N",
    usecols=USECOLS, dtype=DTYPES, low_memory=False,
)
logger.info("title.basics raw: %d rows, %.1f MB",
            len(basics), basics.memory_usage(deep=True).sum() / 1e6)

# numeric coercion (vectorized)
basics["startYear"] = pd.to_numeric(basics["startYear"], errors="coerce")
basics["runtimeMinutes"] = pd.to_numeric(basics["runtimeMinutes"], errors="coerce")
basics["isAdult"] = pd.to_numeric(basics["isAdult"], errors="coerce").fillna(0).astype("int8")

# filter to feature films within sensible bounds
mask = (
    basics["titleType"].isin(settings.IMDB.TITLE_TYPE_FILTER)
    & basics["startYear"].between(settings.IMDB.MIN_YEAR, settings.IMDB.MAX_YEAR)
    & basics["runtimeMinutes"].between(settings.IMDB.MIN_RUNTIME_MIN,
                                       settings.IMDB.MAX_RUNTIME_MIN)
    & (basics["isAdult"] == 0)
)
movies = basics.loc[mask].reset_index(drop=True)
logger.info("After filtering: %d movies (%.1f%% of raw)",
            len(movies), 100 * len(movies) / len(basics))
movies.head()

2026-05-13 21:34:22 | INFO     | 01_data_collection:20 | title.basics raw: 12497885 rows, 1542.8 MB
2026-05-13 21:35:12 | INFO     | 01_data_collection:37 | After filtering: 451754 movies (3.6% of raw)


,tconst,titleType,primaryTitle,originalTitle,isAdult,startYear,runtimeMinutes,genres
0,tt0003854,movie,Dodge City Trail,Dodge City Trail,0,1936,56,"Drama,Music,Western"
1,tt0005076,movie,Charley's Aunt,Charley's Aunt,0,1925,80,Comedy
2,tt0006626,movie,Elnémult harangok,Elnémult harangok,0,1922,52,<NA>
3,tt0008933,movie,Die Brüder Karamasoff,Die Brüder Karamasoff,0,1920,70,Drama
4,tt0010022,movie,The Corsican Brothers,The Corsican Brothers,0,1920,60,"Action,Drama,Romance"


## 5. Join with ratings

`title.ratings` gives us `averageRating` and `numVotes`. We **inner-join** on
`tconst` and additionally require `numVotes >= IMDBConfig.MIN_VOTES` to drop
obscure titles whose ratings are statistically unreliable.

In [7]:
ratings_path = imdb_files["title.ratings.tsv.gz"]

ratings = pd.read_csv(
    ratings_path, sep="\t", na_values="\\N",
    dtype={"tconst": "string", "averageRating": "float32", "numVotes": "int32"},
)
logger.info("title.ratings: %d rows", len(ratings))

movies_rated = movies.merge(ratings, on="tconst", how="inner")
logger.info("After ratings inner-join: %d rows", len(movies_rated))

movies_rated = movies_rated.loc[
    movies_rated["numVotes"] >= settings.IMDB.MIN_VOTES
].reset_index(drop=True)
logger.info("After min-votes filter (>= %d): %d rows",
            settings.IMDB.MIN_VOTES, len(movies_rated))
movies_rated.describe(include="all").T.head(10)

2026-05-13 21:35:16 | INFO     | 01_data_collection:7 | title.ratings: 1669211 rows
2026-05-13 21:35:18 | INFO     | 01_data_collection:10 | After ratings inner-join: 303942 rows
2026-05-13 21:35:18 | INFO     | 01_data_collection:15 | After min-votes filter (>= 1000): 48204 rows


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
tconst,48204,48204,tt0010323,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
titleType,48204,1,movie,48204,NaN,NaN,NaN,NaN,NaN,NaN,NaN
primaryTitle,48204,44413,Mother,9,NaN,NaN,NaN,NaN,NaN,NaN,NaN
originalTitle,48204,45745,Gold,7,NaN,NaN,NaN,NaN,NaN,NaN,NaN
isAdult,48204.0,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0
startYear,48204.0,<NA>,<NA>,<NA>,2001.073998,22.747021,1920.0,1990.0,2009.0,2018.0,2026.0
runtimeMinutes,48204.0,<NA>,<NA>,<NA>,105.268857,22.006399,43.0,91.0,100.0,115.0,299.0
genres,48108,779,Drama,3776,NaN,NaN,NaN,NaN,NaN,NaN,NaN
averageRating,48204.0,NaN,NaN,NaN,6.229211,1.172996,1.0,5.6,6.4,7.1,9.7
numVotes,48204.0,NaN,NaN,NaN,25561.569642,98152.250692,1000.0,1674.0,3326.5,10505.5,3188156.0


## 6. TMDB enrichment — sample

We don't enrich every movie here (that's a long job — done in step 3 of the
project). Instead we **fetch ~20 random samples** to confirm the pipeline
works end-to-end and to give us a feel for what the joined data looks like.

In [8]:
if not settings.TMDB.is_configured():
    print("Skipping TMDB sample — set TMDB_API_KEY in .env to enable.")
else:
    from src.data.tmdb_fetcher import TMDBFetcher

    sample_ids = (
        movies_rated.sort_values("numVotes", ascending=False)
                    .head(20)["tconst"].tolist()
    )
    fetcher = TMDBFetcher()
    sample_df = fetcher.fetch_many(sample_ids)
    sample_df[["imdb_id", "title", "production_countries",
               "budget", "revenue", "popularity",
               "vote_average", "vote_count"]]

Skipping TMDB sample — set TMDB_API_KEY in .env to enable.


## 7. Data-quality report

In [9]:
def quality_report(df: pd.DataFrame, name: str) -> pd.DataFrame:
    null_pct = (df.isna().mean() * 100).round(2)
    duplicates = int(df.duplicated().sum())
    report = pd.DataFrame({
        "dtype":       df.dtypes.astype(str),
        "null_count":  df.isna().sum(),
        "null_pct":    null_pct,
        "n_unique":    df.nunique(dropna=True),
    })
    print(f"=== {name} ===  rows={len(df):,}  duplicates={duplicates}")
    return report

quality_report(movies_rated, "movies (filtered + rated)")

=== movies (filtered + rated) ===  rows=48,204  duplicates=0


,dtype,null_count,null_pct,n_unique
tconst,string,0,0.0,48204
titleType,category,0,0.0,1
primaryTitle,string,0,0.0,44413
originalTitle,string,0,0.0,45745
isAdult,int8,0,0.0,1
startYear,Int64,0,0.0,107
runtimeMinutes,Int64,0,0.0,215
genres,string,96,0.2,779
averageRating,float32,0,0.0,88
numVotes,int32,0,0.0,18688


In [10]:
# Sanity ranges
sanity = pd.DataFrame({
    "startYear":      [movies_rated["startYear"].min(),      movies_rated["startYear"].max()],
    "runtimeMinutes": [movies_rated["runtimeMinutes"].min(), movies_rated["runtimeMinutes"].max()],
    "averageRating":  [movies_rated["averageRating"].min(),  movies_rated["averageRating"].max()],
    "numVotes":       [movies_rated["numVotes"].min(),       movies_rated["numVotes"].max()],
}, index=["min", "max"]).T
sanity

,min,max
startYear,1920.0,2026.0
runtimeMinutes,43.0,299.0
averageRating,1.0,9.7
numVotes,1000.0,3188156.0


In [11]:
# Persist a small "collection summary" alongside the raw files so the
# downstream notebook can verify reproducibility.
import json
summary = {
    "imdb_files":      {k: str(v) for k, v in imdb_files.items()},
    "imdb_row_counts": row_counts,
    "movies_after_filter": int(len(movies)),
    "movies_after_rating_join_and_min_votes": int(len(movies_rated)),
}
out = settings.PATHS.PROCESSED_DATA / "01_collection_summary.json"
out.write_text(json.dumps(summary, indent=2))
logger.info("Wrote %s", out)
summary

2026-05-13 21:35:19 | INFO     | 01_data_collection:12 | Wrote C:\Users\Kadir\Desktop\imdb_project\data\processed\01_collection_summary.json


{'imdb_files': {'title.basics.tsv.gz': 'C:\\Users\\Kadir\\Desktop\\imdb_project\\data\\raw\\title.basics.tsv.gz',
  'title.ratings.tsv.gz': 'C:\\Users\\Kadir\\Desktop\\imdb_project\\data\\raw\\title.ratings.tsv.gz',
  'title.crew.tsv.gz': 'C:\\Users\\Kadir\\Desktop\\imdb_project\\data\\raw\\title.crew.tsv.gz',
  'title.principals.tsv.gz': 'C:\\Users\\Kadir\\Desktop\\imdb_project\\data\\raw\\title.principals.tsv.gz',
  'name.basics.tsv.gz': 'C:\\Users\\Kadir\\Desktop\\imdb_project\\data\\raw\\name.basics.tsv.gz'},
 'imdb_row_counts': {'title.basics.tsv.gz': 12497885,
  'title.ratings.tsv.gz': 1669211,
  'title.crew.tsv.gz': 12497885,
  'title.principals.tsv.gz': 99422049,
  'name.basics.tsv.gz': 15322955},
 'movies_after_filter': 451754,
 'movies_after_rating_join_and_min_votes': 48204}

## ✅ Step 2 outcomes

* IMDB & MovieLens raw files present in `data/raw/`.
* `title.basics` filtered to **feature films** with sensible runtime / year /
  votes guards.
* TMDB client smoke-tested on a small sample.
* A reproducible **collection summary** persisted to
  `data/processed/01_collection_summary.json`.

Next: **Step 3 — Data cleaning & feature engineering** (`02_data_cleaning.ipynb`).